In [1]:
import pandas as pd
import numpy as np

### Dagshub/Mlflow initialization ###

In [52]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='sbolk23', repo_name='House-Prices-Kaggle-Competition', mlflow=True)

Initialized MLflow to track repo "sbolk23/House-Prices-Kaggle-Competition"

Repository sbolk23/House-Prices-Kaggle-Competition initialized!

In [5]:
PATH = './data'
TARGET = 'SalePrice'

In [6]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

### Split Train/Test Data ###

In [70]:
df = pd.read_csv(PATH + '/train.csv')
df = df.drop([523, 1298], axis=0)

In [71]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, shuffle=True, random_state=1337
)

train_ids = X_train['Id']
test_ids = X_test['Id']

print(f'X_train shape {X_train.shape}')
print(f'X_test shape {X_test.shape}')

X_train shape (1166, 80)
X_test shape (292, 80)


### Data Cleaning ###

### Categorical Imputing ###

In [72]:
# we first impute all missing values in categorical and numeric features.

# These categorical columns may have 'NaN' values in them which means missing.
# data_description.txt did not mention 'Nan' was intended.

cat_impute_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'ExterQual',
    'ExterCond',
    'Heating',
    'HeatingQC',
    'CentralAir',
    'Electrical',
    'KitchenQual',
    'Functional',
    'PavedDrive',
    'SaleType',
    'SaleCondition',
]

### Numeric Imputing ###

In [73]:
# We impute all numeric columns because data_description.txt did not mention any numeric feature with intended 'NaN' value.

# num_impute_cols = [
#     'LotFrontage',
#     'GarageYrBlt',
#     'MasVnrArea',
# ]

num_impute_cols = list(X_train.select_dtypes(exclude='object').columns)
num_impute_cols.remove('Id')
num_impute_cols.remove('MSSubClass')

### Categorical One-Hot-Encoded Columns ###

In [74]:
# Here we list features to be one-hot-encoded including 'NaN's because 'NaN' here does not mean missing.
# 'MasVnrType' was interesting column here.

cat_ohe_cols = [
    'MSZoning',
    'Street',
    'Alley',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'GarageType',
    'PavedDrive',
    'MiscFeature',
    'SaleType',
    'SaleCondition',
]

cat_ohe_missing_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'PavedDrive',
    'SaleType',
    'SaleCondition'
]

cat_ohe_absent_cols = [
    'Alley',
    'MasVnrType',
    'GarageType',
    'MiscFeature'
]

### Ordinal Columns ###

In [75]:
# Some categorical columns can may be ordered, replace them with numbers.

cat_ord_cols = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'HeatingQC',
    'KitchenQual',
    'Functional',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

cat_ord_missing_cols = [
    'ExterQual',
    'ExterCond',
    'HeatingQC',
    'KitchenQual',
    'Functional',
]

cat_ord_absent_cols = [
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

exter_qual_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
exter_cond_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_qual_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_cond_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_exposure_order     = ['NA', 'No', 'Mn', 'Av', 'Gd']
bsmt_fin_type_1_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
bsmt_fin_type_2_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
heating_qc_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
kitchen_qual_order      = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
functional_order        = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
fireplace_qu_order      = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_finish_order     = ['NA', 'Unf', 'RFn', 'Fin']
garage_qual_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_cond_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
pool_qc_order           = ['NA', 'Fa', 'TA', 'Gd', 'Ex']
fence_order             = ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

cat_ord_missing_categories = [
    exter_qual_order,
    exter_cond_order,
    heating_qc_order,
    kitchen_qual_order,
    functional_order,
]

cat_ord_absent_categories = [
    bsmt_qual_order,
    bsmt_cond_order,
    bsmt_exposure_order,
    bsmt_fin_type_1_order,
    bsmt_fin_type_2_order,
    fireplace_qu_order,
    garage_finish_order,
    garage_qual_order,
    garage_cond_order,
    pool_qc_order,
    fence_order,
]

### Numeric One-Hot-Encoded Columns ###

In [76]:
# This feature has numeric type which is misleading.

num_ohe_cols = [
    'MSSubClass',
]

### Data Cleanup Pipeline ###

In [77]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

num_impute_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='median')),
    ('scale',   StandardScaler())
])

num_ohe_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_ord_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=cat_ord_missing_categories)),
])

cat_ord_absent_pipeline = Pipeline([
    ('nan_impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='NA')),
    ('ordinal',     OrdinalEncoder(categories=cat_ord_absent_categories)),
])

cat_ohe_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

cat_ohe_absent_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='Missing')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num_impute',       num_impute_pipeline,       num_impute_cols),
        ('num_ohe',          num_ohe_pipeline,          num_ohe_cols),
        ('cat_ord_absent',   cat_ord_absent_pipeline,   cat_ord_absent_cols),
        ('cat_ord_missing',  cat_ord_missing_pipeline,  cat_ord_missing_cols),
        ('cat_ohe_absent',   cat_ohe_absent_pipeline,   cat_ohe_absent_cols),
        ('cat_ohe_missing',  cat_ohe_missing_pipeline,  cat_ohe_missing_cols),
        ('drop',    'drop',                             ['Id']),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

preprocessor.set_output(transform='pandas')

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_impute', ...), ('num_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [78]:
# scaling every column after preprocessing decreased accuracy.
# scaling only numeric columns is sufficient.

preprocessor_pipeline = Pipeline([
    ('preprocessor', preprocessor),
])

preprocessor_pipeline.set_output(transform='pandas')

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_impute', ...), ('num_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

In [79]:
y_train_t = np.log1p(y_train)
y_test_t = np.log1p(y_test)

preprocessor_pipeline.fit(X_train, y_train_t)

X_train_t = preprocessor_pipeline.transform(X_train)
X_test_t = preprocessor_pipeline.transform(X_test)

X = X_train.copy()
X['SalePrice'] = y_train

kf = KFold(n_splits=5, shuffle=True, random_state=1337)
folds = list(kf.split(X))

train_idx, val_idx = folds[4]
vals = X.iloc[train_idx, :]
outliers_in_fold = vals[(vals['GrLivArea'] > 4000) & (vals['SalePrice'] < 300000)]
outliers_in_fold[['GrLivArea', 'SalePrice']]

,GrLivArea,SalePrice


### Full Pipeline (Linear Regression) ###

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold, ParameterGrid

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', LinearRegression())
])

param_grid = {
    'preprocessor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring=['neg_root_mean_squared_error'],
    refit='neg_root_mean_squared_error',
    return_train_score=True,
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)

### MLFlow Log (Linear Regression) ###

In [59]:
mlflow.set_experiment('linear_regression')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    with mlflow.start_run(run_name=f'LR_{i}'):
        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('data_version', 'v1_with_outliers')

        # Here we log preprocessing hyperparameters



        # Here we log model parameters
        for key, value in row_dict.items():
            if 'mean' in key or 'std' in key or 'split' in key:
                mlflow.log_metric(key, value)
            elif 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)


#  preprocessing_spec = {
#     'num_impute_cols': num_impute_cols,
#     'num_ohe_cols': num_ohe_cols,
#     'cat_ord_absent_cols': cat_ord_absent_cols,
#     'cat_ord_missing_cols': cat_ord_missing_cols,
#     'cat_ohe_absent_cols': cat_ohe_absent_cols,
#     'cat_ohe_missing_cols': cat_ohe_missing_cols,
#     'cat_ord_absent_categories': cat_ord_absent_categories,
#     'cat_ord_missing_categories': cat_ord_missing_categories,
# }

🏃 View run LR_0 at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/06cd80b588d84324b574e6083617929d
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
🏃 View run LR_1 at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/2186e82c9ae24867ba522d650331e363
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1


### Full Pipeline (Ridge) ###

In [80]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge())
])

param_grid = {
    'preprocessor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
    # 'model__alpha': [.001, .01, .05, .1, .5, 1, 5, 9, 9.1, 9.2, 9.3, 9.4],
    # 'model__alpha': np.linspace(.01, 50, 300),
    'model__alpha': np.logspace(np.log10(9), np.log10(10), 300),
}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)

cv_results_df

Fitting 5 folds for each of 600 candidates, totalling 3000 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,param_preprocessor__preprocessor__num_impute__impute__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.362533,0.100524,0.129788,0.024883,9.000000,mean,"{'model__alpha': 9.0, 'preprocessor__preproces...",-0.136602,-0.122212,-0.108522,-0.098003,-0.113638,-0.115795,0.013025,600,-0.088444,-0.093136,-0.094647,-0.097346,-0.093460,-0.093407,0.002890
1,0.319033,0.091877,0.149912,0.059834,9.000000,median,"{'model__alpha': 9.0, 'preprocessor__preproces...",-0.136611,-0.122184,-0.108517,-0.097999,-0.113644,-0.115791,0.013026,588,-0.088441,-0.093135,-0.094652,-0.097348,-0.093460,-0.093407,0.002892
2,0.376756,0.090122,0.142200,0.032727,9.003172,mean,"{'model__alpha': 9.003171945585345, 'preproces...",-0.136601,-0.122211,-0.108521,-0.098004,-0.113638,-0.115795,0.013024,599,-0.088445,-0.093138,-0.094649,-0.097347,-0.093461,-0.093408,0.002890
3,0.309720,0.040450,0.133818,0.051803,9.003172,median,"{'model__alpha': 9.003171945585345, 'preproces...",-0.136610,-0.122183,-0.108516,-0.097999,-0.113644,-0.115791,0.013026,586,-0.088443,-0.093137,-0.094654,-0.097350,-0.093462,-0.093409,0.002892
4,0.334137,0.089902,0.141364,0.049566,9.006345,mean,"{'model__alpha': 9.006345009086111, 'preproces...",-0.136601,-0.122210,-0.108520,-0.098004,-0.113638,-0.115795,0.013024,598,-0.088446,-0.093139,-0.094651,-0.097349,-0.093462,-0.093409,0.002890
5,0.281448,0.067110,0.139147,0.043728,9.006345,median,"{'model__alpha': 9.006345009086111, 'preproces...",-0.136610,-0.122182,-0.108515,-0.098000,-0.113644,-0.115790,0.013026,584,-0.088444,-0.093138,-0.094655,-0.097351,-0.093463,-0.093410,0.002892
6,0.321296,0.075895,0.117328,0.020946,9.009519,mean,"{'model__alpha': 9.009519190896297, 'preproces...",-0.136600,-0.122209,-0.108520,-0.098004,-0.113638,-0.115794,0.013024,597,-0.088448,-0.093141,-0.094652,-0.097350,-0.093464,-0.093411,0.002890
7,0.279010,0.031564,0.110830,0.011148,9.009519,median,"{'model__alpha': 9.009519190896297, 'preproces...",-0.136609,-0.122181,-0.108515,-0.098000,-0.113644,-0.115790,0.013025,582,-0.088445,-0.093140,-0.094657,-0.097353,-0.093464,-0.093412,0.002892
8,0.258029,0.022542,0.108355,0.006749,9.012694,mean,"{'model__alpha': 9.012694491410032, 'preproces...",-0.136600,-0.122208,-0.108519,-0.098004,-0.113638,-0.115794,0.013024,596,-0.088449,-0.093142,-0.094654,-0.097351,-0.093465,-0.093412,0.002890
9,0.282638,0.031193,0.113219,0.003744,9.012694,median,"{'model__alpha': 9.012694491410032, 'preproces...",-0.136609,-0.122180,-0.108514,-0.098000,-0.113644,-0.115789,0.013025,580,-0.088447,-0.093141,-0.094658,-0.097354,-0.093465,-0.093413,0.002892


### Full Pipeline (Lasso) ###

In [123]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Lasso(max_iter=10000))
])

param_grid = {
    'preprocessor__preprocessor__num_impute__impute__strategy': ['mean', 'median'],
    # 'model__alpha': [.0005, .0006, .0007, .0008, .0009, .001, .005, .01, .05, .1, .5, 1, 5, 10],
    'model__alpha': np.logspace(-4, 1, 300),
}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)

cv_results_df

Fitting 5 folds for each of 600 candidates, totalling 3000 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,param_preprocessor__preprocessor__num_impute__impute__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.798562,0.288849,0.192542,0.053236,0.000100,mean,"{'model__alpha': 0.0001, 'preprocessor__prepro...",-0.156672,-0.126751,-0.205040,-0.131516,-0.151069,-0.154210,0.027816,184,-0.091772,-0.092366,-0.093926,-0.093994,-0.088841,-0.092180,0.001881
1,0.494709,0.080717,0.153101,0.040209,0.000100,median,"{'model__alpha': 0.0001, 'preprocessor__prepro...",-0.156628,-0.126733,-0.205042,-0.131520,-0.151075,-0.154200,0.027819,183,-0.091784,-0.092372,-0.093927,-0.093998,-0.088841,-0.092184,0.001882
2,0.605070,0.101625,0.159681,0.027988,0.000104,mean,"{'model__alpha': 0.00010392556829367131, 'prep...",-0.156585,-0.126356,-0.205113,-0.130929,-0.150560,-0.153909,0.028027,178,-0.091921,-0.092515,-0.094042,-0.094112,-0.089000,-0.092318,0.001865
3,0.627594,0.091575,0.162277,0.035550,0.000104,median,"{'model__alpha': 0.00010392556829367131, 'prep...",-0.156541,-0.126338,-0.205119,-0.130930,-0.150566,-0.153899,0.028031,177,-0.091933,-0.092521,-0.094042,-0.094116,-0.089000,-0.092322,0.001866
4,0.411241,0.058485,0.140560,0.040918,0.000108,mean,"{'model__alpha': 0.00010800523745162529, 'prep...",-0.156482,-0.125957,-0.205170,-0.130335,-0.150103,-0.153609,0.028232,172,-0.092078,-0.092674,-0.094143,-0.094230,-0.089164,-0.092458,0.001845
5,0.460329,0.098281,0.147776,0.047190,0.000108,median,"{'model__alpha': 0.00010800523745162529, 'prep...",-0.156440,-0.125939,-0.205177,-0.130338,-0.150107,-0.153600,0.028237,171,-0.092090,-0.092680,-0.094143,-0.094234,-0.089163,-0.092462,0.001846
6,0.462331,0.079025,0.160286,0.045095,0.000112,mean,"{'model__alpha': 0.0001122450568085307, 'prepr...",-0.156337,-0.125553,-0.205220,-0.129715,-0.149639,-0.153293,0.028441,168,-0.092240,-0.092846,-0.094238,-0.094355,-0.089328,-0.092601,0.001825
7,0.455558,0.073185,0.119823,0.008281,0.000112,median,"{'model__alpha': 0.0001122450568085307, 'prepr...",-0.156304,-0.125536,-0.205228,-0.129724,-0.149644,-0.153287,0.028445,167,-0.092252,-0.092852,-0.094238,-0.094359,-0.089328,-0.092606,0.001826
8,0.471180,0.173373,0.122983,0.003217,0.000117,mean,"{'model__alpha': 0.00011665131316981962, 'prep...",-0.156127,-0.125137,-0.205257,-0.129080,-0.149141,-0.152949,0.028650,164,-0.092409,-0.093030,-0.094335,-0.094490,-0.089469,-0.092747,0.001817
9,0.557206,0.164909,0.121148,0.003691,0.000117,median,"{'model__alpha': 0.00011665131316981962, 'prep...",-0.156096,-0.125121,-0.205262,-0.129090,-0.149147,-0.152943,0.028652,163,-0.092422,-0.093036,-0.094336,-0.094495,-0.089469,-0.092751,0.001817


### Final Pipeline (ElasticNet) ###

In [81]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', ElasticNet(max_iter=10000))
])

param_grid = {
    'preprocessor__preprocessor__num_impute__impute__strategy':
        ['mean', 'median'],
    # 'model__alpha':
    #     [.0005, .001, .002, .005, .1, .5, 1],
    # 'model__l1_ratio':
    #     [.1, .2, .3, .4, .5, .6, .7, .8, .9],

    'model__alpha':
        np.logspace(-4, 1, 30),
    'model__l1_ratio':
        np.linspace(.05, .95, 20)
}

# {\'model__alpha\': 0.0007278953843983154, \'model__l1_ratio\': 0.7131578947368421, \'preprocessor__preprocessor__num_impute__impute__strategy\': \'mean\'}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1,
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)
cv_results_df

Fitting 5 folds for each of 1200 candidates, totalling 6000 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,param_model__l1_ratio,param_preprocessor__preprocessor__num_impute__impute__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,3.314577,3.392029,0.261353,0.102276,0.000100,0.050000,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.142375,-0.137402,-0.130484,-0.106842,-0.124139,-0.128248,0.012357,569,-0.081123,-0.083706,-0.085303,-0.089008,-0.085955,-0.085019,0.002599
1,3.442399,2.941844,0.192308,0.057177,0.000100,0.050000,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.142391,-0.137390,-0.130474,-0.106837,-0.124179,-0.128254,0.012358,570,-0.081118,-0.083703,-0.085311,-0.089008,-0.085951,-0.085018,0.002601
2,2.970581,2.475001,0.202630,0.048384,0.000100,0.097368,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.142077,-0.136683,-0.129734,-0.106614,-0.123047,-0.127631,0.012311,557,-0.081194,-0.083776,-0.085358,-0.089078,-0.086018,-0.085085,0.002598
3,2.901912,2.743442,0.165322,0.021030,0.000100,0.097368,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.142090,-0.136672,-0.129723,-0.106611,-0.123089,-0.127637,0.012310,558,-0.081189,-0.083773,-0.085365,-0.089079,-0.086013,-0.085084,0.002600
4,2.308816,2.204081,0.221125,0.096675,0.000100,0.144737,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.141703,-0.135960,-0.129092,-0.106259,-0.122393,-0.127081,0.012265,547,-0.081276,-0.083859,-0.085419,-0.089161,-0.086085,-0.085160,0.002596
5,2.324319,1.797121,0.164558,0.032120,0.000100,0.144737,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.141714,-0.135950,-0.129075,-0.106256,-0.122427,-0.127085,0.012265,548,-0.081271,-0.083857,-0.085426,-0.089162,-0.086081,-0.085159,0.002598
6,0.782089,0.147618,0.160000,0.036521,0.000100,0.192105,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.141331,-0.135335,-0.128693,-0.105867,-0.121816,-0.126608,0.012251,541,-0.081369,-0.083954,-0.085487,-0.089250,-0.086162,-0.085245,0.002593
7,0.700925,0.133491,0.168572,0.035241,0.000100,0.192105,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.141340,-0.135326,-0.128675,-0.105863,-0.121848,-0.126610,0.012250,542,-0.081364,-0.083952,-0.085495,-0.089251,-0.086159,-0.085244,0.002595
8,0.659668,0.188697,0.134308,0.012748,0.000100,0.239474,mean,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.140978,-0.134776,-0.128283,-0.105502,-0.121283,-0.126164,0.012238,525,-0.081472,-0.084059,-0.085558,-0.089341,-0.086237,-0.085334,0.002587
9,0.550341,0.162798,0.135884,0.024841,0.000100,0.239474,median,"{'model__alpha': 0.0001, 'model__l1_ratio': 0....",-0.140987,-0.134767,-0.128265,-0.105499,-0.121315,-0.126167,0.012237,526,-0.081467,-0.084056,-0.085566,-0.089342,-0.086234,-0.085333,0.002589


In [121]:
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', ElasticNet(alpha=0.0007278953843983154, l1_ratio=0.7131578947368421))
])
final_pipeline.fit(X_train, y_train_t)
y_pred = final_pipeline.predict(X_test)
print(root_mean_squared_error(y_test_t, y_pred))

y_pred = final_pipeline.predict(X_train)
print(root_mean_squared_error(y_train_t, y_pred))



# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', Lasso(alpha=.0007934096665797492))
# ])
# final_pipeline.fit(X_train, y_train_t)
# y_pred = final_pipeline.predict(X_test)
# print(root_mean_squared_error(y_test_t, y_pred))
#
# y_pred = final_pipeline.predict(X_train)
# print(root_mean_squared_error(y_train_t, y_pred))
#
#
# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', LinearRegression())
# ])
# final_pipeline.fit(X_train, y_train_t)
# y_pred = final_pipeline.predict(X_test)
# print(root_mean_squared_error(y_test_t, y_pred))
#
# y_pred = final_pipeline.predict(X_train)
# print(root_mean_squared_error(y_train_t, y_pred))

0.11544915376776345
0.10881045553882318
0.1156369745205326
0.11243910374712052
0.12966410509994647
0.09419088615888568


In [189]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge(alpha=400))
])

cv_results = cross_validate(
    final_pipeline, X_train, y_train_t,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
)

cv_results_df = pd.DataFrame(cv_results)

train_root_mean_squared_error = -cv_results_df.mean()['train_score']
validation_root_mean_squared_error = -cv_results_df.mean()['test_score']

print(f'train_rmse: {train_root_mean_squared_error}')
print(f'validation_rmse: {validation_root_mean_squared_error}')

train_rmse: 0.10310912196089887
validation_rmse: 0.13957445547139208


In [124]:
# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', ElasticNet(alpha=0.0007278953843983154, l1_ratio=0.7131578947368421))
# ])

# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', Ridge(alpha=9.846791241561434))
# ])

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Lasso(alpha=0.0006348565216305217))
])

test = pd.read_csv(PATH + '/test.csv')
test_ids = test['Id']

# final_pipeline.fit(pd.concat([X_train, X_test], axis=0, ignore_index=True), pd.concat([y_train_t, y_test_t], axis=0, ignore_index=True))
final_pipeline.fit(X_train, y_train_t)
prices_log = final_pipeline.predict(test)
prices = np.expm1(prices_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': prices,
})

submission.to_csv('submission.csv', index=False)